In [1]:
import pandas as pd

df_new = pd.read_csv("data/raw/pd_2018_present.csv")
df_old = pd.read_csv('data/raw/pd_2003_2018.csv')

In [2]:
print(f"Columns in new: {df_new.columns}")
print(f"Columns in old: {df_old.columns}")


Columns in new: Index(['Date', 'Time', 'Police District', 'Resolution', 'Latitude',
       'Longitude', 'Location', 'category_harmonized'],
      dtype='str')
Columns in old: Index(['PdId', 'IncidntNum', 'Incident Code', 'Category', 'Descript',
       'DayOfWeek', 'Date', 'Time', 'PdDistrict', 'Resolution', 'Address', 'X',
       'Y', 'location', 'data_loaded_at'],
      dtype='str')


In [3]:
mapping = {
    'IncidntNum': 'Incident Number',
    'Category': 'Incident Category',
    'Date': 'Incident Date',
    'Time': 'Incident Time',
    'PdDistrict': 'Police District',
    'X': 'Latitude',
    'Y': 'Longitude'
}

df_old = df_old.rename(columns=mapping)
df_old = df_old[
            ['Incident Number', 
             'Incident Date', 'Incident Time', 'Incident Code', 
             'Incident Category','Police District', 'Latitude', 'Longitude']
            ]

In [4]:
df_old['Row ID'] = df_old.index
df_old['Incident Datetime'] = df_old['Incident Date'] + " " + df_old['Incident Time']
df_old['Incident Datetime'] = pd.to_datetime(df_old['Incident Datetime'], format='%m/%d/%Y %H:%M')
df_new['Incident Datetime'] = pd.to_datetime(df_new['Incident Datetime'], format='%Y/%m/%d %I:%M:%S %p')


KeyError: 'Incident Datetime'

In [ ]:
df_new = df_new.drop(df_new[['Incident Subcategory']], axis=1)
df_new.info()


In [ ]:
df_old.info()

In [ ]:
# Make Police districts regular letters, not capital
df_old['Police District'].unique, df_new['Police District'].unique

df_new['Police District'] = pd.Categorical(df_new['Police District'])
df_old['Police District'] = pd.Categorical(df_old['Police District'])

df_new['Police District'] = df_new['Police District'].str.upper()
df_old['Police District'].unique(), df_new['Police District'].unique()


df_new['Incident Category'] = pd.Categorical(df_new['Incident Category'])
df_old['Incident Category'] = pd.Categorical(df_old['Incident Category'])

In [ ]:
df_old['Latitude'].head(), df_new['Latitude'].head()

In [ ]:
print(len(df_old['Incident Category'].unique()), len(df_new['Incident Category'].unique()))

new_list = sorted(df_old['Incident Category'].astype(str).unique())
old_list = sorted(df_new['Incident Category'].astype(str).unique())

for i in range(0, len(old_list)-1):
    print(f"{old_list[i]}")
    # if i <= len(old_list)-1:
    #     print(f"    {new_list[i]}")#, {old_list[i]}")
    # else:
    #     print(f"NEW:    {new_list[i]}")

In [ ]:
# make both category names capitalized
df_old['Incident Category'] = pd.Categorical(df_old['Incident Category'])
df_old['Incident Category'] = df_old['Incident Category'].map(lambda x: x.lower(), na_action=None)

df_new['Incident Category'] = pd.Categorical(df_new['Incident Category'])
df_new['Incident Category'] = df_new['Incident Category'].map(lambda x: x.lower(), na_action=None)

In [ ]:
mappings = {
    # THEFT
    "larceny/theft": "theft",
    "larceny theft": "theft",
    "stolen property": "theft",
    "lost property": "theft",


    # VIOLENCE
    "assault": "assault",
    "robbery": "assault",
    "homicide": "assault",
    "kidnapping": "assault",

    # Vehicle theft
    "motor vehicle theft": "vehicle theft",
    "motor vehicle theft?": "vehicle theft",
    "vehicle theft": "vehicle theft",
    "vehicle misplaced": "vehicle theft",
    "vehicle impounded": "vehicle theft",
    "recovered vehicle": "vehicle theft",

    # DRUGS
    "drug/narcotic": "drug",
    "drug offense": "drug",
    "drug violation": "drug",

    # SEX OFFENSES
    "sex offenses, forcible": "sex_offense",
    "sex offenses, non forcible": "sex_offense",
    "rape": "sex_offense",
    "sex offense": "sex_offense",
    "human trafficking (a), commercial sex acts": "sex_offense",
    "human trafficking (b), involuntary servitude": "sex_offense",
    "human trafficking, commercial sex acts": "sex_offense",
    "prostitution": "sex_offense",

    # WEAPONS
    "weapon laws": "weapons",
    "weapons offense": "weapons",
    "weapons offence": "weapons",
    "weapons carrying etc": "weapons",

    # FRAUD / FINANCIAL
    "forgery/counterfeiting": "fraud",
    "forgery and counterfeiting": "fraud",
    "fraud": "fraud",
    "bad checks": "fraud",
    "bribery": "fraud",
    "extortion": "fraud",

    # ADMIN / OTHER
    "case closure": "other",
    "courtesy report": "other",
    "miscellaneous investigation": "other",
    "other": "other",
    "other miscellaneous": "other",
    "civil sidewalks": "other",
    "secondary codes": "other",
    "trea": "other",
    "trespass": "other",
    "suspicious": "other",
    "suspicious occ": "other",
    "offences against the family and children": "other",
    "fire report": "other",
    "liquor laws": "other",
    "gambling": "other",
    "missing person": "other",
    "vandalism": "other",
    "civil sidewalks": "other",
    "fire report": "other",
    "offences against the family and children": "other",
    "vehicle impounded": "other",
    "vehicle misplaced": "other",
    "suspicious": "other",
    "other offences": "other",
        "Other offenses": "other",
}


#print(df_old['Incident Category'].unique())
df_old['Incident Category'] = df_old['Incident Category'].map(mappings, na_action=None).fillna(df_old['Incident Category'])
print(df_old['Incident Category'].unique())


In [ ]:
print(df_new['Incident Category'].unique())

In [ ]:
focus_crime = [
        'RECOVERED VEHICLE', 'ASSAULT', 'FRAUD', 'LARCENY THEFT', 
        'LOST PROPERTY', 'HUMAN TRAFFICKING, COMMERCIAL SEX ACTS',
        'ARSON', 'ASSAULT', 'BURGLARY']
        
df = pd.concat([df_old, df_new], axis=0)
df = df[df['Incident Category'].isin(focus_crime)]
df.info()

In [ ]:
# Now we fetch Incident Numbers where there are duplicates
keys = ['Incident Number', 'Incident Datetime', 'Incident Code', 'Police District']

print("Before cleaning: ", df.shape)
df_cleaned = df[df.duplicated(subset=keys) == False]

print("After cleaning: ", df_cleaned.shape)

# grouped = df_filtered.groupby(['Incident Number', 'Incident Datetime']).filter(lambda x: x.count() != 1)
# grouped

In [ ]:
df_cleaned.to_csv('data/processed/cleaned_2003_present.csv')

In [ ]:
df_plot = (
    df[['Incident Datetime', 'Incident Category']]
        .assign(Year=lambda x: x['Incident Datetime'].dt.year)
        .groupby(['Year', 'Incident Category']).agg('count')
        .unstack('Incident Category', fill_value=0)
        .sort_index()
)
df_plot


In [ ]:
df_plot.index

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math

# plot
cols = df_plot.columns
years = df_plot.index

labels =[c[1] if isinstance(c, tuple) else str(c) for c in cols]
n = len(labels)

# Setup
n = len(labels)
ncols = 2
nrows = math.ceil(n/ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 2.8 * nrows), sharex=True, sharey=True)
axes = axes.ravel()  # make it 1D so we can index easily

# Plot for each type of crime
for ax, col, label in zip(axes, cols, labels):
       ax.plot(years, df_plot[col],marker="o", linewidth=1)
       ax.set_title(label)
       ax.grid(True, alpha=0.3)
       
for ax in axes[::2]:
       ax.set_ylabel('No of Incidents')
       
for ax in axes[6:]:
       ax.set_xlabel('Years')
       
fig.tight_layout()
       
plt.show()